In [ ]:
# ============================================================
# CONFIGURATION - every path comes from config/paths.py, the single
# source of truth. Override cluster locations with the MUSICA_ENV_*
# environment variables documented there. Do not hard-code paths here.
# ============================================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
_ROOT = next(p for p in [_here, *_here.parents]
             if (p / 'config' / 'paths.py').exists())
sys.path.insert(0, str(_ROOT))
import config  # also puts functions/ on sys.path
from config import paths as P


This script is used to analyze the MUSICA simulations for the CONUS_BackgroundO3 

In [ ]:
figure_diri = f'{P.FIGURES_ROOT}/CESM_analysis/BGO3/'

### Download figures to local disk: 
# rsync -avz --exclude=".*" -e ssh "<username>@svante9.mit.edu:{P.FIGURES_ROOT}/CESM_analysis/BGO3/*" "/Users/$USER/Downloads/"
# rsync -avz --exclude=".*" -e ssh "<username>@svante9.mit.edu:{P.FIGURES_ROOT}/CESM_analysis/BGO3/*" "/Users/$USER/Downloads/同步空间/MUSICA(CESM22-SE)/Y2022_BackgroundO3/Figures/"

### Functions

In [ ]:
import os
import glob
import fnmatch

import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point, Polygon

import xarray as xr
import numpy as np

import matplotlib.pyplot as plt # Core library for plotting
import matplotlib.cm as cm # To use different colormaps
import cartopy.crs as ccrs # For map projection
import seaborn as sns # boxplot

from matplotlib.cm import ScalarMappable


In [ ]:
import sys
# functions I defined
sys.path.insert(0,f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/')
from Plot_2D import Plot_2D # To draw a map
from func_MUSICA_DefineRegion import *

SCRIP_CONUS = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons.nc'


In [ ]:
def extract_date_correctly(filename):
    # Extracting the date portion from filename based on observed structure
    date_section = filename.split('.')[-2].split('-')
    # Combine the first three elements to form the date in YYYY-MM-DD
    full_date = '-'.join(date_section[:3])
    return full_date


In [ ]:
# Specify timezone for the given region
def regional_UTCtimezone_offset_summer(fileregion):
    """This function returns the timezone of a given region
       *With daylight saving  
       Options: WestCoast,Mountain,Midwest,Southwest,Southeast,Northeast
    """
    # Define regions        
    if fileregion == "WestCoast":
        # Pacific (UTC-7)
        UTCtimezone_offset = -7
    elif fileregion == "Mountain":
        # Mountain (UTC-6)
        UTCtimezone_offset = -6
    elif fileregion == "Midwest":
        # Central (UTC-5)
        UTCtimezone_offset = -5
    elif fileregion == "Southwest":
        # Central (UTC-5)
        UTCtimezone_offset = -5
    elif fileregion == "Southeast":
        # Eastern (UTC-4)
        UTCtimezone_offset = -4
    elif fileregion == "Northeast":
        # Eastern (UTC-4)
        UTCtimezone_offset = -4
    
    return UTCtimezone_offset

In [ ]:
varlabel_dic = {'O3':r'$O_{3}$',
                'NO2':r'$NO_{2}$',
                'Ox':r'$O_{x}$',
                'NOx':r'$NO_{x}$',
                'HCHO':r'$HCHO$', # listing twice if pointed
               'CH2O':r'$HCHO$', #r'$CH_{2}O$'
                'CO':r'$CO$',
                'SO2':r'$SO_{2}$',
                'PM25':r'$PM_{2.5}$',
               }

unit_dic = {'O3':r'$ppb$',
                'NO2':r'$ppb$',
            'Ox':r'$ppb$',
                'NOx':r'$ppb$',
            'HCHO':r'$ppb$', # listing twice if pointed
               'CH2O':r'$ppb$',
                'CO':r'$ppb$',
                'SO2':r'$ppb$',
                'PM25':r'$\mu g/m^{3}$',   #r'$kg/m^{3}$', 
               }

Scalefactor_dic = {'O3':1e9,
                'NO2':1e9,
                'Ox':1e9,
                'NOx':1e9,
               'CH2O':1e9,
                'CO':1e9,
                'SO2':1e9,
                'PM25':1e9,
               }

In [ ]:
PerturbedAEmis = [
    'NO','NH3','CO',
    'C2H2','C2H4','C2H5OH','C2H6','C3H6','C3H8','CH2O',
    'CH3CHO','CH3COCH3','CH3OH', 
    
    'ISOP','MEK','MTERP',
    'BENZENE','BIGENE','TOLUENE',
    'SVOC',
    'SO2',
    'bc_a4','num_bc_a4','num_pom_a4','pom_a4'
]

In [ ]:
region_bounds = {
        "CONUS": {"lon_range": [-140, -50], "lat_range": [15, 60]},
        "Global": {"lon_range": None, "lat_range": None},
        "NYC": {"lon_range": [-74.5, -73.5], "lat_range": [40.4, 41]},
        "ExtendedNYC": {"lon_range": [-75, -72], "lat_range": [40, 42]}
    }

# Monthly Mean Simulations

In [ ]:
svante_archive = f'{P.ARCHIVE}/'

# list for all case names
BASE2022_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20220401TY20230401'
BASE2023_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.BASEY20230401TY20231101'
noBB2022_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noBBemisCONUS80kmBufferY20220401TY20221101'
noBB2023_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noBBemisCONUS80kmBufferY20230401TY20231101'
noAnthro2022_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noANTHROemisCONUS80kmBufferY20220401TY20221101'
noAnthro2023_casename = 'f.e22.FCnudged.ne30_ne30_mg17.BGO3.noANTHROemisCONUS80kmBufferY20230401TY20231101'

fileregion_ls = ['WestCoast','Mountain','Midwest','Southwest','Northeast','Southeast']
nfileregion = len(fileregion_ls)

# label for each casename
casename_label_dic = {
                'SLAMS':'SLAMS',
                BASE2022_casename:'BASE2022',
                BASE2023_casename:'BASE2023',
                ### Perturbations
                noBB2022_casename:'noBB2022',
                noBB2023_casename:'noBB2023',
                noAnthro2022_casename:'noAnthro2022',
                noAnthro2023_casename:'noAnthro2023',
                }


## Differences across years

In [ ]:
# Plot differences in surface O3 for monthly means
lev_idx = -1

Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

BASEYear1 = '2022'
BASEYear2 = '2023'


# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    if subploti<7:
        # Get the corresponding subplot
        ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allyearcase_monthi_ds_dic = {}
        BASEYear1_casename = BASE2022_casename
        BASEYear2_casename = BASE2023_casename

        for casenamei in [BASEYear1_casename,BASEYear2_casename]:
            if casenamei==BASEYear1_casename:
                pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{BASEYear1}-{Monthi}.nc'
            elif casenamei==BASEYear2_casename:
                pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{BASEYear2}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allyearcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        BASEYear2_minus_BASEYear1_da = allyearcase_monthi_ds_dic[BASEYear2_casename][varname]-allyearcase_monthi_ds_dic[BASEYear1_casename][varname]

        Time1 = f'{BASEYear1}T{Monthi}'
        Time2 = f'{BASEYear2}T{Monthi}'

        Plot_ar = BASEYear2_minus_BASEYear1_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 10 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, state=True, lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{Time2} minus \n{Time1} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    else:
        # ax.axis('off')
        fig.delaxes(ax)
        
axes.flat[-1].set_visible(False)    

# Overall title
plttitle = f'Interannual BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}InterannualBASEDiff_04T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()  

In [ ]:
# Plot differences in surface O3 for monthly means
lev_idx = -1

Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

BASEYear1 = '2022'
BASEYear2 = '2023'


# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    if subploti<7:
        # Get the corresponding subplot
        ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allyearcase_monthi_ds_dic = {}
        BASEYear1_casename = BASE2022_casename
        BASEYear2_casename = BASE2023_casename

        for casenamei in [BASEYear1_casename,BASEYear2_casename]:
            if casenamei==BASEYear1_casename:
                pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{BASEYear1}-{Monthi}.nc'
            elif casenamei==BASEYear2_casename:
                pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{BASEYear2}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allyearcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        BASEYear2_minus_BASEYear1_da = allyearcase_monthi_ds_dic[BASEYear2_casename][varname]-allyearcase_monthi_ds_dic[BASEYear1_casename][varname]

        Time1 = f'{BASEYear1}T{Monthi}'
        Time2 = f'{BASEYear2}T{Monthi}'

        Plot_ar = BASEYear2_minus_BASEYear1_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 10 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, 
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{Time2} minus \n{Time1} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    else:
        # ax.axis('off')
        fig.delaxes(ax)
        
axes.flat[-1].set_visible(False)    

# Overall title
plttitle = f'Interannual BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}InterannualBASEDiff_04T10_Global.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()  

#### Draft

In [ ]:
# Plot differences in surface O3 for monthly means => Read in data
lev_idx = -1

Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

BASEYear1 = '2022'
BASEYear2 = '2023'

### Calculate the difference across years for the same month
for Monthi in Months_ls: 
    allyearcase_monthi_ds_dic = {}
    BASEYear1_casename = BASE2022_casename
    BASEYear2_casename = BASE2023_casename
    
    for casenamei in [BASEYear1_casename,BASEYear2_casename]:
        if casenamei==BASEYear1_casename:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{BASEYear1}-{Monthi}.nc'
        elif casenamei==BASEYear2_casename:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{BASEYear2}-{Monthi}.nc'

        # Read in
        casei_ds = xr.open_dataset(pathi)
        selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
        allyearcase_monthi_ds_dic[casenamei] = selcasei_ds
        
    ### Plot
    # Plot the difference | O3
    setmap = 'bwr'
    PlotRegion = "CONUS" 

    scalefactor = Scalefactor_dic[varname]
    # calculate the difference
    BASEYear2_minus_BASEYear1_da = allyearcase_monthi_ds_dic[BASEYear2_casename][varname]-allyearcase_monthi_ds_dic[BASEYear1_casename][varname]

    Time1 = f'{BASEYear1}T{Monthi}'
    Time2 = f'{BASEYear2}T{Monthi}'

    Plot_ar = BASEYear2_minus_BASEYear1_da.values*scalefactor
    Plot_unit = unit_dic[varname]

    rangeMax = 10 #np.nanmean(Plot_ar) #setvmax*scalefactor
    rangeMin = -rangeMax

    ### Which map
    fig = plt.figure( figsize=(8,6) ) 
    # - ne30x8 regional refinement over CONUS|
    ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=rangeMin, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, 
                 lon_range=region_bounds[PlotRegion]['lon_range'], 
                 lat_range=region_bounds[PlotRegion]['lat_range'],
              grid_line=False, grid_line_lw=0.15 ) 

    # titlestr = f'{longname} | BASE\n {Timei[:13]}'
    titlestr = f'{varname} | BASE\n {Time2} minus {Time1} Mean'
    # plt.title(longname++' \n'+Timei[:13], fontsize=16, y=1.02);
    plt.title(titlestr, fontsize=16, y=1.02);

    np.nanmean(Plot_ar)

In [ ]:
allyearcase_monthi_ds_dic.keys()

## Differences across scenarios in the same year

In [ ]:
# Plot differences in surface O3 for monthly means | noANTHRO
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2022_casename, 
                   noAnthro2022_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noAnthro2022_minus_BASE2022_da = allcase_monthi_ds_dic[noAnthro2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noAnthro2022_minus_BASE2022_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 30 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        # settitle = f'{varlabel_dic[varname]} | noANTHRO-BASE\n {Timei} Mean'
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)    

# Overall title
plttitle = f'noANTHRO-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noANTHRO-BASE_202204T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot differences in surface O3 for monthly means | noANTHRO | Global
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2022_casename, 
                   noAnthro2022_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noAnthro2022_minus_BASE2022_da = allcase_monthi_ds_dic[noAnthro2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noAnthro2022_minus_BASE2022_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 30 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, 
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        # settitle = f'{varlabel_dic[varname]} | noANTHRO-BASE\n {Timei} Mean'
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)    

# Overall title
plttitle = f'noANTHRO-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noANTHRO-BASE_202204T10_Global.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot differences in surface O3 for monthly means | noANTHRO
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2023_casename, 
                   noAnthro2023_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noAnthro2023_minus_BASE2023_da = allcase_monthi_ds_dic[noAnthro2023_casename][varname]-allcase_monthi_ds_dic[BASE2023_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noAnthro2023_minus_BASE2023_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 30 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        # settitle = f'{varlabel_dic[varname]} | noANTHRO-BASE\n {Timei} Mean'
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)    

# Overall title
plttitle = f'noANTHRO-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noANTHRO-BASE_202304T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot differences in surface O3 for monthly means | noANTHRO
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2023_casename, 
                   noAnthro2023_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noAnthro2023_minus_BASE2023_da = allcase_monthi_ds_dic[noAnthro2023_casename][varname]-allcase_monthi_ds_dic[BASE2023_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noAnthro2023_minus_BASE2023_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 30 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, 
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        # settitle = f'{varlabel_dic[varname]} | noANTHRO-BASE\n {Timei} Mean'
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)    

# Overall title
plttitle = f'noANTHRO-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noANTHRO-BASE_202304T10_Global.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot differences in surface O3 for monthly means | noBB
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2022_casename, 
                   noBB2022_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2022_minus_BASE2022_da = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2022_minus_BASE2022_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)     


# Overall title
plttitle = f'noBB-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noBB-BASE_202204T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot differences in surface O3 for monthly means | noBB
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2022_casename, 
                   noBB2022_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2022_minus_BASE2022_da = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2022_minus_BASE2022_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax,
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)     

# Overall title
plttitle = f'noBB-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noBB-BASE_202204T10_Global.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot differences in surface O3 for monthly means | noBB
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2023_casename, 
                   noBB2023_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2023_minus_BASE2023_da = allcase_monthi_ds_dic[noBB2023_casename][varname]-allcase_monthi_ds_dic[BASE2023_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2023_minus_BASE2023_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)     


# Overall title
plttitle = f'noBB-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noBB-BASE_202304T10_CONUS.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

In [ ]:
# Plot differences in surface O3 for monthly means | noBB
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.15}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2023_casename, 
                   noBB2023_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2023_minus_BASE2023_da = allcase_monthi_ds_dic[noBB2023_casename][varname]-allcase_monthi_ds_dic[BASE2023_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2023_minus_BASE2023_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, 
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)     

# Overall title
plttitle = f'noBB-BASE | {varlabel_dic[varname]}'
fig.suptitle(plttitle, fontsize=28, y=1.02, fontweight='bold')

savefig_filename = f'{figure_diri}noBB-BASE_202304T10_Global.png'
# Save the figure
plt.savefig(savefig_filename, dpi=300, bbox_inches='tight')  # Adjust the filename and dpi as needed
print(f'Save to {savefig_filename}')
plt.show()

#### Draft

In [ ]:
# Plot differences in surface O3 for monthly means
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']


# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.25}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2022_casename, 
                   noBB2022_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2022_minus_BASE2022_da = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2022_minus_BASE2022_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, #state=True,
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{varlabel_dic[varname]} | noBB-BASE\n {Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

In [ ]:
# Plot differences in surface O3 for monthly means | 2023
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']


# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.25}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2023_casename, 
                   noBB2023_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2023_minus_BASE2023_da = allcase_monthi_ds_dic[noBB2023_casename][varname]-allcase_monthi_ds_dic[BASE2023_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2023_minus_BASE2023_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{varlabel_dic[varname]} | noBB-BASE\n {Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

In [ ]:
# Plot differences in surface O3 for monthly means | 2023
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']


# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.25}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2023_casename, 
                   noBB2023_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2023_minus_BASE2023_da = allcase_monthi_ds_dic[noBB2023_casename][varname]-allcase_monthi_ds_dic[BASE2023_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2023_minus_BASE2023_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up], 
                     lon_range=[-140,-50], lat_range=[15,60],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{varlabel_dic[varname]} | noBB-BASE\n {Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

In [ ]:
# Plot differences in surface O3 for monthly means | 2023
lev_idx = -1

Yeari = '2023'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']


# Map setting
setcmap = cm.RdBu_r  # for difference maps
setunit_size, settitle_size, settitle_size2, setcolortick_size = [40, 25, 49, 35]

###-----------------------------------
### Plot a x rows by x columns map
nrows, ncols = 2, 4

# Create a new figure with subplots sharing x and y axes
fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(5.9 * ncols, 3.64 * nrows), 
    sharex=True, 
    sharey=True, 
    subplot_kw={'projection': ccrs.PlateCarree()},  # Set projection for all subplots
    gridspec_kw={'wspace': 0, 'hspace': 0.25}  # Remove space between subplots
)
# fig.tight_layout(pad=0)  # Remove padding

# Define the extent for all subplots
# Select for a given region
fileregion = 'CONUS'
lon_right,lat_bot,lon_left,lat_up = latlonbound(fileregion)
lon_range = [lon_left, lon_right]
lat_range = [lat_bot, lat_up] 

varname = 'O3'
scalefactor = Scalefactor_dic[varname]
setunit = unit_dic[varname]

for subploti in range(len(Months_ls)):
    ax = axes.flat[subploti]
    ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
    if subploti<=6:
        # Get the corresponding subplot
        # ax.set_extent([lon_range[0], lon_range[1], lat_range[0], lat_range[1]], crs=ccrs.PlateCarree())
        # Get the data in the corresponding month
        Monthi = Months_ls[subploti]
        allcase_monthi_ds_dic = {}
        case_ls = [BASE2023_casename, 
                   noBB2023_casename]

        for casenamei in case_ls:
            pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'

            # Read in
            casei_ds = xr.open_dataset(pathi)
            # print(pathi)
            selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
            allcase_monthi_ds_dic[casenamei] = selcasei_ds
            
        # calculate the difference
        noBB2023_minus_BASE2023_da = allcase_monthi_ds_dic[noBB2023_casename][varname]-allcase_monthi_ds_dic[BASE2023_casename][varname]
        
        Timei = f'{Yeari}T{Monthi}'

        Plot_ar = noBB2023_minus_BASE2023_da.values*scalefactor
        Plot_unit = unit_dic[varname]

        rangeMax = 3 #np.nanmean(Plot_ar) #setvmax*scalefactor
        
        # show colorbar for the last row in each column 
        im = Plot_2D(Plot_ar, scrip_file=SCRIP_CONUS, ax=ax,
                    unit=setunit,unit_size=setunit_size,colortick_size=setcolortick_size,
                    cmin=-rangeMax, cmax=rangeMax, #state=True, 
                     # lon_range=[lon_right, lon_left], lat_range=[lat_bot, lat_up],
                     cmap=setcmap, colorbar=False, 
                    grid_line=False)
        
        
        # # add title label
        settitle = f'{varlabel_dic[varname]} | noBB-BASE\n {Timei} Mean'
        # settitle = f'{caseDiffi_label}'
        ax.set_title(f'{settitle}',c='k',fontsize=settitle_size,y=1.02, fontweight='bold'
                    # bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.01'),y=0.87,
                    )        
        # If this is the last column, add a colorbar
        # if subploti == 3:
        if subploti == 6:
            norm = plt.Normalize(vmin=-rangeMax, vmax=rangeMax)
            # Create a dummy color-mappable object for the colorbar
            dummy_mappable = ScalarMappable(norm=norm, cmap=setcmap)
            dummy_mappable.set_array([])  # Required for colorbar creation

            # fig.add_axes([left, bottom, width, height])
            cax = fig.add_axes([ax.get_position().x1+0.002, ax.get_position().y0, 0.01, ax.get_position().height*1])
            cbar = fig.colorbar(dummy_mappable, cax=cax, orientation='vertical', extend='both') #extend='both'
            cbar.set_label(setunit, fontsize=setunit_size)
            cbar.ax.tick_params(labelsize=setcolortick_size)
        
        # Hide x and y-axis ticks and labels
        ax.set_xticks([])
        ax.set_yticks([])
        
    elif subploti>6:
        # ax.axis('off')
        # ax.set_axis_off()
        ax.set_visible(False) 
        
axes.flat[-1].set_visible(False)        

In [ ]:
# Plot differences in surface O3 for monthly means
lev_idx = -1

Yeari = '2022'
Months_ls = ['04','05','06','07','08','09','10']
var_ls = ['O3']

for Monthi in Months_ls: 
    allcase_monthi_fullpath_dic = {}
    allcase_monthi_ds_dic = {}
    case_ls = [BASE2022_casename, 
               noBB2022_casename]

    for casenamei in case_ls:
        pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'
        allcase_monthi_fullpath_dic[casenamei] = pathi

        # Read in
        casei_ds = xr.open_dataset(pathi)
        print(pathi)
        selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
        allcase_monthi_ds_dic[casenamei] = selcasei_ds
    
    ### Plot
    # Plot the difference | O3
    setmap = 'bwr'
    PlotRegion = "CONUS" 

    scalefactor = Scalefactor_dic[varname]
    # calculate the difference
    noBB2022_minus_BASE2022_da = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]

    Timei = f'{Yeari}T{Monthi}' #str(Plot_da.time.values)

    Plot_ar = noBB2022_minus_BASE2022_da.values*scalefactor
    Plot_unit = unit_dic[varname]

    rangeMax = 3 #np.nanmean(Plot_ar) #setvmax*scalefactor
    rangeMin = -rangeMax

    ### Which map
    fig = plt.figure( figsize=(8,6) ) 
    # - ne30x8 regional refinement over CONUS|
    ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=rangeMin, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, 
                 lon_range=region_bounds[PlotRegion]['lon_range'], 
                 lat_range=region_bounds[PlotRegion]['lat_range'],
              grid_line=False, grid_line_lw=0.15 ) 

    # titlestr = f'{longname} | BASE\n {Timei[:13]}'
    titlestr = f'{varname} | noBB-BASE\n {Timei} Mean'
    # plt.title(longname++' \n'+Timei[:13], fontsize=16, y=1.02);
    plt.title(titlestr, fontsize=16, y=1.02);

    np.nanmean(Plot_ar)

#### Test with one month

In [ ]:
### Read in monthly means for surface
lev_idx = -1

Yeari = '2022'
Monthi = '07'
var_ls = ['O3','NO','NO2']
allcase_monthi_fullpath_dic = {}
allcase_monthi_ds_dic = {}
case_ls = [BASE2022_casename, 
           noBB2022_casename]

for casenamei in case_ls:
    pathi = f'{svante_archive}{casenamei}/atm/hist/{casenamei}.cam.h0.{Yeari}-{Monthi}.nc'
    allcase_monthi_fullpath_dic[casenamei] = pathi
    
    # Read in
    casei_ds = xr.open_dataset(pathi)
    print(pathi)
    selcasei_ds = casei_ds.isel(lev=lev_idx,time=0)[var_ls]
    allcase_monthi_ds_dic[casenamei] = selcasei_ds
    

In [ ]:
varname = 'O3'
allcase_monthi_ds_dic[BASE2022_casename][varname]

In [ ]:
varname = 'O3'

# calculate the difference
noBB2022_minus_BASE2022_ds = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]

In [ ]:
noBB2022_minus_BASE2022_ds

In [ ]:
SCRIP_CONUS

In [ ]:
# Plot the difference | O3
setmap = 'bwr'
PlotRegion = "CONUS" 

scalefactor = Scalefactor_dic[varname]
# calculate the difference
noBB2022_minus_BASE2022_da = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]

Timei = f'{Yeari}T{Monthi}' #str(Plot_da.time.values)

Plot_ar = noBB2022_minus_BASE2022_da.values*scalefactor
Plot_unit = [varname]

rangeMax = 3 #np.nanmean(Plot_ar) #setvmax*scalefactor
rangeMin = -rangeMax

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
        cmin=rangeMin, cmax=rangeMax, cmap=setmap, 
        unit=Plot_unit,
        state=True, 
             lon_range=region_bounds[PlotRegion]['lon_range'], 
             lat_range=region_bounds[PlotRegion]['lat_range'],
          grid_line=False, grid_line_lw=0.15 ) 

# titlestr = f'{longname} | BASE\n {Timei[:13]}'
titlestr = f'{varname} | noBB-BASE\n {Timei} Mean'
# plt.title(longname++' \n'+Timei[:13], fontsize=16, y=1.02);
plt.title(titlestr, fontsize=16, y=1.02);

np.nanmean(Plot_ar)

In [ ]:
# Plot the BASE case over NYC | O3
setmap = 'bwr' #'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}"

# surface O3
varname = 'O3'
scalefactor = 1e9
# calculate the difference
noBB2022_minus_BASE2022_da = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]

Timei = f'{Yeari}T{Monthi}' #str(Plot_da.time.values)

Plot_ar = noBB2022_minus_BASE2022_da.values*scalefactor
Plot_unit = 'ppb' #Plot_da.units#+' ('+formatted_label+')'
# longname = Plot_da.long_name

rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
rangeMin = -rangeMax

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
        cmin=rangeMin, cmax=rangeMax, cmap=setmap, 
        unit=Plot_unit,
        state=True, 
          grid_line=False, grid_line_lw=0.15 ) 

# titlestr = f'{longname} | BASE\n {Timei[:13]}'
titlestr = f'{varname} | noBB-BASE\n {Timei} Mean'
# plt.title(longname++' \n'+Timei[:13], fontsize=16, y=1.02);
plt.title(titlestr, fontsize=16, y=1.02);

np.nanmean(Plot_ar)


In [ ]:
# Plot the BASE case| O3
setmap = 'YlOrRd' #'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = "CONUS" #'ExtendedNYC'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}"

# surface O3
varname = 'O3'
scalefactor = 1e9

Timei = f'{Yeari}T{Monthi}' #str(Plot_da.time.values)

Plot_ar = allcase_monthi_ds_dic[BASE2022_casename][varname].values*scalefactor
Plot_unit = 'ppb' #Plot_da.units#+' ('+formatted_label+')'
# longname = Plot_da.long_name

rangeMax = 70 #np.nanmean(Plot_ar) #setvmax*scalefactor
rangeMin = 0

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
        cmin=rangeMin, cmax=rangeMax, cmap=setmap, 
        unit=Plot_unit,
        state=True, 
             lon_range=region_bounds[PlotRegion]['lon_range'], 
             lat_range=region_bounds[PlotRegion]['lat_range'],
          grid_line=False, grid_line_lw=0.15 ) 

# titlestr = f'{longname} | BASE\n {Timei[:13]}'
titlestr = f'{varname} | BASE\n {Timei} Mean'
# plt.title(longname++' \n'+Timei[:13], fontsize=16, y=1.02);
plt.title(titlestr, fontsize=16, y=1.02);

np.nanmean(Plot_ar)

In [ ]:
# Plot the BASE case| O3
setmap = 'YlOrRd' #'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = "CONUS" #'ExtendedNYC'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}"

# surface O3
varname = 'O3'
scalefactor = 1e9

Timei = f'{Yeari}T{Monthi}' #str(Plot_da.time.values)

Plot_ar = allcase_monthi_ds_dic[noBB2022_casename][varname].values*scalefactor
Plot_unit = 'ppb' #Plot_da.units#+' ('+formatted_label+')'
# longname = Plot_da.long_name

rangeMax = 70 #np.nanmean(Plot_ar) #setvmax*scalefactor
rangeMin = 0

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
        cmin=rangeMin, cmax=rangeMax, cmap=setmap, 
        unit=Plot_unit,
        state=True, 
             lon_range=region_bounds[PlotRegion]['lon_range'], 
             lat_range=region_bounds[PlotRegion]['lat_range'],
          grid_line=False, grid_line_lw=0.15 ) 

# titlestr = f'{longname} | BASE\n {Timei[:13]}'
titlestr = f'{varname} | noBB\n {Timei} Mean'
# plt.title(longname++' \n'+Timei[:13], fontsize=16, y=1.02);
plt.title(titlestr, fontsize=16, y=1.02);

np.nanmean(Plot_ar)

In [ ]:
# Plot the BASE case over NYC | O3
setmap = 'bwr' #'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = "CONUS" #'ExtendedNYC'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}"

# surface O3
varname = 'NO2'
scalefactor = 1e9
# calculate the difference

noBB2022_minus_BASE2022_da = allcase_monthi_ds_dic[noBB2022_casename][varname]-allcase_monthi_ds_dic[BASE2022_casename][varname]

Timei = f'{Yeari}T{Monthi}' #str(Plot_da.time.values)

Plot_ar = noBB2022_minus_BASE2022_da.values*scalefactor
Plot_unit = 'ppb' #Plot_da.units#+' ('+formatted_label+')'
# longname = Plot_da.long_name

rangeMax = 5 #np.nanmean(Plot_ar) #setvmax*scalefactor
rangeMin = -rangeMax

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
        cmin=rangeMin, cmax=rangeMax, cmap=setmap, 
        unit=Plot_unit,
        state=True, 
             lon_range=region_bounds[PlotRegion]['lon_range'], 
             lat_range=region_bounds[PlotRegion]['lat_range'],
          grid_line=False, grid_line_lw=0.15 ) 

# titlestr = f'{longname} | BASE\n {Timei[:13]}'
titlestr = f'{varname} | noBB-BASE\n {Timei} Mean'
# plt.title(longname++' \n'+Timei[:13], fontsize=16, y=1.02);
plt.title(titlestr, fontsize=16, y=1.02);

np.nanmean(Plot_ar)

# Other